> **Public archive note.** Earlier checkpoint-grid experiment notebook. Outputs removed. Run only with original private or licensed input datasets, prepared scripts and checkpoint data; not a standalone exact replay.


In [ ]:

# KAGGLE SETUP — HANDLES EXTRACTED FOLDERS OR ZIP FILES

from pathlib import Path
from zipfile import ZipFile
import shutil, subprocess, sys, json, hashlib, torch

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

TRAIN_NAME = "crisis_consolidated_humanitarian_filtered_lang_en_train.tsv"
TEST_NAME = "crisis_consolidated_humanitarian_filtered_lang_en_test.tsv"

# 1. FIND ORIGINAL DATASETS
train_files = list(INPUT.rglob(TRAIN_NAME))
test_files = list(INPUT.rglob(TEST_NAME))

print("Training TSV files:", len(train_files))
print("Testing TSV files:", len(test_files))

assert len(train_files) == 1, "Expected one original training TSV"
assert len(test_files) == 1, "Expected one original testing TSV"

TRAIN = train_files[0]
TEST = test_files[0]

# 2. FIND THE RESUMABLE TRAINING CODE
script_files = list(INPUT.rglob("run_experiment_resumable.py"))

if script_files:
    print("Using already extracted training code.")
    SOURCE_ROOT = script_files[0].parent.parent

else:
    # Fallback: find any ZIP containing the expected script
    SOURCE_ROOT = None
    code_dir = WORK / "vircon_source"

    for archive in INPUT.rglob("*.zip"):
        with ZipFile(archive) as z:
            if any(
                name.endswith("scripts/run_experiment_resumable.py")
                for name in z.namelist()
            ):
                code_dir.mkdir(parents=True, exist_ok=True)
                z.extractall(code_dir)
                matches = list(
                    code_dir.rglob("run_experiment_resumable.py")
                )
                assert len(matches) == 1
                SOURCE_ROOT = matches[0].parent.parent
                break

    assert SOURCE_ROOT is not None, (
        "Resumable trainer not found in the attached dataset."
    )

# Copy code to writable Kaggle working directory
ROOT = WORK / "vircon_source_runtime"

if not (ROOT / "scripts/run_experiment_resumable.py").exists():
    shutil.copytree(SOURCE_ROOT, ROOT, dirs_exist_ok=True)

TRAIN_SCRIPT = ROOT / "scripts/run_experiment_resumable.py"
GRID_SCRIPT = ROOT / "scripts/stage2_checkpoint_grid.py"
FOLDS = (
    ROOT / "data_provenance/event_disjoint_fold_assignment.csv"
)

for p in [TRAIN_SCRIPT, GRID_SCRIPT, FOLDS]:
    assert p.is_file(), f"Required file missing: {p}"

# 3. VERIFY ORIGINAL DATASET HASHES
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1048576), b""):
            h.update(chunk)
    return h.hexdigest()

assert sha256(TRAIN) == (
    "5e3dd75d1dcbdba893d8d731795b51853d04708547a9bdc2bd18a0b54b82541b"
), "Training dataset checksum mismatch"

assert sha256(TEST) == (
    "a664ce57f40c8791228d5a00ab43c23965c2f1be90fde4307413217fca8d1a86"
), "Testing dataset checksum mismatch"

# 4. RESTORE COMPLETED EXPERIMENTS AND CHECKPOINTS
GRID_OUT = WORK / "Checkpointed_Grid"

def finished_count(folder):
    count = 0
    for f in folder.rglob("SUCCESS.json"):
        try:
            if json.loads(f.read_text()).get("completed") is True:
                count += 1
        except Exception:
            pass
    return count

if not GRID_OUT.exists():
    sources = [
        p for p in INPUT.rglob("Checkpointed_Grid")
        if p.is_dir()
    ]

    assert sources, "Original checkpoint grid not found"

    if len(sources) > 1:
        print("Multiple saved grids found:")
        for p in sources:
            print(p, "completed:", finished_count(p))
        raise RuntimeError(
            "Select the latest saved checkpoint grid before training."
        )

    shutil.copytree(sources[0], GRID_OUT)

# 5. INSTALL REQUIREMENTS
REQ = ROOT / "requirements-colab.txt"

assert REQ.is_file(), f"Missing requirements: {REQ}"

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "-q", "-r", str(REQ)
])

# 6. REPORT STATUS
def finished_runs():
    result = []
    for f in GRID_OUT.rglob("SUCCESS.json"):
        try:
            if json.loads(f.read_text()).get("completed") is True:
                result.append(f.parent)
        except Exception:
            pass
    return result

completed = len(finished_runs())

print("\n" + "=" * 60)
print("KAGGLE MIGRATION SUCCESSFUL")
print("=" * 60)
print("Training data verified:", TRAIN.is_file())
print("Testing data verified:", TEST.is_file())
print("Training script:", TRAIN_SCRIPT)
print("Ablation runner:", GRID_SCRIPT)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Completed experiments restored:", completed)
print("Remaining experiments:", 120 - completed)
print("Resume checkpoints:", len(list(
    GRID_OUT.rglob("checkpoints/latest.pt")
)))

print("\nREADY FOR TRAINING")


In [ ]:

import subprocess, sys, time

# Start conservatively: two experiments per Kaggle session.
MAX_RUNS_THIS_SESSION = 2

command = [
    sys.executable,
    str(GRID_SCRIPT),
    "--train-tsv", str(TRAIN),
    "--test-tsv", str(TEST),
    "--fold-assignment", str(FOLDS),
    "--output", str(GRID_OUT),
    "--folds", "1,2,3,4,5",
    "--seeds", "20266009,20266010,20266008",
    "--variants",
    "full,no_event,no_contrastive,no_request,"
    "no_triage,distilbert,crisis,tfidf_svc",
    "--epochs", "2",
    "--batch-size", "24",
    "--checkpoint-every-steps", "100",
    "--max-runs", "1"
]

for i in range(MAX_RUNS_THIS_SESSION):
    before = len(finished_runs())

    if before >= 120:
        print("All planned experiments are complete.")
        break

    print(f"\nStarting next run; {before}/120 completed")
    start = time.time()

    subprocess.check_call(command)

    after = len(finished_runs())
    print("Completed:", after, "/120")
    print("Elapsed minutes:", round((time.time()-start)/60, 1))

    if after <= before:
        print("No new completion found. Check logs before continuing.")
        break

print("\nSESSION FINISHED")
print("Completed:", len(finished_runs()))
print("Remaining:", 120 - len(finished_runs()))


In [ ]:

import json
from pathlib import Path

GRID = Path("/kaggle/working/Checkpointed_Grid")

completed = []

for p in sorted(GRID.rglob("SUCCESS.json")):
    try:
        status = json.loads(p.read_text())
        if status.get("completed") is True:
            completed.append(p.parent)
    except Exception as e:
        print("Unreadable status:", p, e)

print("IJIES STAGE 2 STATUS")
print("=" * 60)
print("Completed experiments:", len(completed))
print("Remaining experiments:", 120 - len(completed))

print("\nLATEST CHECKPOINT STATUS")
for p in sorted(GRID.rglob("checkpoints/latest.pt")):
    print("Checkpoint:", p.parent.parent.relative_to(GRID))

print("\nCRISIS ENCODER EXPERIMENT")
run = (
    GRID / "fold1" /
    "event-dev_fold1_crisis_seed20266009"
)

success = run / "SUCCESS.json"
summary = run / "summary.json"

print("Completed:", success.exists())

if summary.exists():
    results = json.loads(summary.read_text())
    print("Macro F1:", results.get("macro_f1"))
    print("Accuracy:", results.get("accuracy"))
    print(
        "Balanced accuracy:",
        results.get("balanced_accuracy")
    )
else:
    print("No final evaluation summary yet.")


In [ ]:

from pathlib import Path
import pandas as pd
import json

GRID = Path("/kaggle/working/Checkpointed_Grid")

records = []

for success in sorted(GRID.rglob("SUCCESS.json")):
    folder = success.parent
    summary = folder / "summary.json"

    if not summary.exists():
        continue

    try:
        status = json.loads(success.read_text())

        if status.get("completed") is not True:
            continue

        metrics = json.loads(summary.read_text())

        records.append({
            "experiment": str(folder.relative_to(GRID)),
            "macro_f1": metrics.get("macro_f1"),
            "accuracy": metrics.get("accuracy"),
            "balanced_accuracy": metrics.get("balanced_accuracy"),
            "n_eval": metrics.get("n", metrics.get("n_eval"))
        })

    except Exception as e:
        print("Read error:", folder, e)

df = pd.DataFrame(records)

if not df.empty:
    df = df.sort_values("experiment").reset_index(drop=True)
    display(df)

    df.to_csv(
        "/kaggle/working/IJIES_9_Runs_Results.csv",
        index=False
    )

print("Total completed:", len(df))
print("Remaining:", 120 - len(df))


In [ ]:

import pandas as pd
from pathlib import Path

assignment = pd.read_csv(FOLDS)

print("ARCHIVED FOLD ASSIGNMENTS")
print("=" * 65)
print("Columns:", list(assignment.columns))
print("Total rows:", len(assignment))

fold_col = next(
    (c for c in assignment.columns
     if c.lower() in ["fold", "fold_id", "outer_fold",
                       "test_fold", "eval_fold"]),
    None
)
event_col = next(
    (c for c in assignment.columns
     if c.lower() in ["canonical_event", "event",
                       "event_id", "event_name"]),
    None
)

if fold_col:
    print("\nRows assigned to each fold:")
    print(assignment[fold_col].value_counts().sort_index())

    if event_col:
        print("\nDistinct events per fold:")
        print(
            assignment.groupby(fold_col)[event_col]
            .nunique()
        )

        events_per_fold = (
            assignment.groupby(event_col)[fold_col].nunique()
        )
        conflicts = events_per_fold[events_per_fold > 1]
        print("Events appearing in multiple held-out folds:",
              len(conflicts))

        if len(conflicts):
            print("WARNING: Investigate these event assignments.")
else:
    print("Fold column not identified; inspect columns above.")

print("\nSAVED MODEL EVALUATION")
print("=" * 65)

for fold in [1, 2]:
    run = (
        GRID_OUT / f"fold{fold}" /
        f"event-dev_fold{fold}_full_seed20266009"
    )

    pred_file = run / "predictions_no_text.csv"

    if pred_file.exists():
        pred = pd.read_csv(pred_file)
        print(f"\nFold {fold}: {len(pred)} evaluation rows")

        pred_event_col = next(
            (c for c in pred.columns
             if c.lower() in ["event", "canonical_event",
                               "event_id", "event_name"]),
            None
        )

        if pred_event_col:
            print(
                "Unique evaluation events:",
                pred[pred_event_col].nunique()
            )
            print(
                pred[pred_event_col].value_counts().head(10)
            )

print("\nVALIDATION REPORT COMPLETE")


In [ ]:

# PRIORITY EXPERIMENTS:
# Full VIRCON-ERT, Crisis Encoder, DistilBERT
# Across all 5 event-disjoint folds
# Seed 20266009

import subprocess
import sys
import torch

assert torch.cuda.is_available(), "Enable Kaggle GPU."

PRIORITY_COMMAND = [
    sys.executable,
    str(GRID_SCRIPT),
    "--train-tsv", str(TRAIN),
    "--test-tsv", str(TEST),
    "--fold-assignment", str(FOLDS),
    "--output", str(GRID_OUT),
    "--folds", "1,2,3,4,5",
    "--seeds", "20266009",
    "--variants", "full,crisis,distilbert",
    "--epochs", "2",
    "--batch-size", "24",
    "--checkpoint-every-steps", "250",
    "--max-runs", "1"
]

print("Completed before starting:", len(finished_runs()))
print("Starting next priority experiment...")

subprocess.check_call(PRIORITY_COMMAND)

print("Completed after execution:", len(finished_runs()))
print("Save your Kaggle notebook outputs before ending the session.")
